# Choosing representative runs: test examples

Shows what `select_runs.py` chooses on a few typical cases.
It takes the runs given by `ena_runs()` and keeps at most 10, as different as possible, with simple rules:
1. remove the runs with no FASTQ file, less than 1 billion bases, or whose material is not the species' own tissue RNA (microbes = METATRANSCRIPTOMIC / METAGENOMIC, or single cell);
2. inside each tissue, paired-end runs first, then the deepest;
3. one run per tissue, in rounds, each time from a study not used yet if possible, and never two runs of the same study and tissue (replicates).

Only ENA's search answer is read; no reads file is downloaded.

In [1]:
import sys
sys.path.append("../../..")

from backend.raw_data.ena_runs import ena_runs
from backend.raw_data.select_runs import select_runs


def show(choice):
    """Print how many runs were given, kept and chosen, then one line per chosen run with the reason."""
    print("given:", choice["runs_given"], "| useful:", choice["runs_useful"], "| selected:", len(choice["selected"]))
    for run in choice["selected"]:
        print(run["run_accession"], run["study_accession"], run["layout"],
              round(run["base_count"] / 1e9, 1), "Gb |", run["reason"])
    studies = {run["study_accession"] for run in choice["selected"]}
    tissues = {run["tissue_type"] for run in choice["selected"]}
    print("different studies:", len(studies), "| different tissues:", len(tissues))

## 1. A species with many studies (cat)

The 2,000 RNA-seq runs of the cat come from 93 studies. After removing the small runs and the microbe and single-cell runs, 1,541 are left.
The 10 chosen runs have 10 different tissues and 9 different studies: only the pancreas run comes from a study already used (for the heart), but for a new tissue.
The tissues studied by the most teams come first (kidney, heart, lung...).
Careful: the kidney run comes from a "Single cell atlas" study, but ENA marks it as normal TRANSCRIPTOMIC, so the filter can't see it.

In [2]:
cat = ena_runs(9685)
show(select_runs(cat["runs"]))

given: 2000 | useful: 1541 | selected: 10
SRR15199661 PRJNA747757 PAIRED 86.9 Gb | round 1: tissue 'kidney', from a study not used yet
SRR3200471 PRJNA312519 PAIRED 26.9 Gb | round 1: tissue 'heart', from a study not used yet
SRR35438978 PRJNA1330231 PAIRED 55.1 Gb | round 1: tissue 'lung', from a study not used yet
SRR21657517 PRJNA882763 PAIRED 13.7 Gb | round 1: tissue 'liver', from a study not used yet
SRR23085858 PRJNA924061 PAIRED 18.5 Gb | round 1: tissue 'skin', from a study not used yet
DRR487133 PRJDB15955 PAIRED 13.1 Gb | round 1: tissue 'small intestine', from a study not used yet
SRR14251502 PRJNA721786 PAIRED 18.5 Gb | round 1: tissue 'blood', from a study not used yet
SRR3200469 PRJNA312519 PAIRED 20.8 Gb | round 1: tissue 'pancreas', from a study already used, but not for this tissue
SRR2495945 PRJNA296557 PAIRED 169.5 Gb | round 1: tissue 'brain', from a study not used yet
SRR11939416 PRJNA637678 PAIRED 12.5 Gb | round 1: tissue 'spleen', from a study not used yet
diff

## 2. A species with fewer studies (Egyptian fruit bat)

One of the bats test cases. 116 of its 361 runs are single cell and are removed, so 187 runs are left.
There are fewer studies than tissues, so most runs come from a study already used, but for a new tissue (see the reasons): 10 tissues from only 4 studies.

In [3]:
bat = ena_runs(9407)
show(select_runs(bat["runs"]))

given: 361 | useful: 187 | selected: 10
SRR2914360 PRJNA300284 PAIRED 17.9 Gb | round 1: tissue 'kidney', from a study not used yet
SRR35797469 PRJNA1344459 SINGLE 58.5 Gb | round 1: tissue 'bone marrow', from a study not used yet
SRR2914368 PRJNA300284 PAIRED 22.1 Gb | round 1: tissue 'axillary lymph nodes', from a study already used, but not for this tissue
SRR2914295 PRJNA300284 PAIRED 18.8 Gb | round 1: tissue 'brain', from a study already used, but not for this tissue
SRR16761642 PRJNA777475 PAIRED 57.3 Gb | round 1: tissue 'caudate nucleus of the basal ganglia', from a study not used yet
SRR2914359 PRJNA300284 PAIRED 5.0 Gb | round 1: tissue 'heart', from a study already used, but not for this tissue
SRR2914059 PRJNA300284 PAIRED 22.5 Gb | round 1: tissue 'liver', from a study already used, but not for this tissue
SRR2913598 PRJNA300284 PAIRED 19.4 Gb | round 1: tissue 'lung', from a study already used, but not for this tissue
SRR32671429 PRJNA1235284 PAIRED 10.3 Gb | round 1: ti

## 3. Changing the settings

The number of runs and the minimum depth can be changed. Here: 5 runs, at least 20 billion bases each.

In [4]:
show(select_runs(cat["runs"], max_runs=5, min_bases=20_000_000_000))

given: 2000 | useful: 185 | selected: 5
SRR15199664 PRJNA747757 PAIRED 85.1 Gb | round 1: tissue 'heart', from a study not used yet
SRR35439013 PRJNA1330231 PAIRED 52.7 Gb | round 1: tissue 'liver', from a study not used yet
SRR15199660 PRJNA747757 PAIRED 88.8 Gb | round 1: tissue 'lung', from a study already used, but not for this tissue
SRR34712201 PRJNA1295995 PAIRED 50.4 Gb | round 1: tissue 'kidney cortex', from a study not used yet
SRR34712186 PRJNA1295995 PAIRED 44.1 Gb | round 1: tissue 'kidney medulla', from a study already used, but not for this tissue
different studies: 3 | different tissues: 5


## 4. Fewer useful runs than asked

With a high minimum (50 billion bases), only 12 bat runs are left. Most of them are replicates (same study and same tissue), which would add almost nothing.
The script never takes two runs of the same study and tissue, so it stops with only 4 runs instead of 10.
An empty list gives an empty selection.

In [5]:
show(select_runs(bat["runs"], min_bases=50_000_000_000))
print()
show(select_runs([]))

given: 361 | useful: 12 | selected: 4
SRR35797469 PRJNA1344459 SINGLE 58.5 Gb | round 1: tissue 'bone marrow', from a study not used yet
SRR16761642 PRJNA777475 PAIRED 57.3 Gb | round 1: tissue 'caudate nucleus of the basal ganglia', from a study not used yet
SRR16761646 PRJNA777475 PAIRED 54.2 Gb | round 1: tissue 'primary motor cortex, orofacial region (ofm1)', from a study already used, but not for this tissue
SRR16761650 PRJNA777475 PAIRED 60.0 Gb | round 1: tissue 'primary motor cortex, wing region (wm1)', from a study already used, but not for this tissue
different studies: 2 | different tissues: 4

given: 0 | useful: 0 | selected: 0
different studies: 0 | different tissues: 0


## 5. Known limit: very big runs

Because the deepest run of each tissue comes first, the chosen runs can be very big compared to the usual depth of the species (to check with the supervisors).

In [6]:
depths = sorted(run["base_count"] / 1e9 for run in cat["runs"] if run["base_count"])
chosen = [run["base_count"] / 1e9 for run in select_runs(cat["runs"])["selected"]]
print("usual depth of a cat run (middle):", round(depths[len(depths) // 2], 1), "Gb")
print("depth of the chosen runs:", sorted(round(d, 1) for d in chosen))

usual depth of a cat run (middle): 6.6 Gb
depth of the chosen runs: [12.5, 13.1, 13.7, 18.5, 18.5, 20.8, 26.9, 55.1, 86.9, 169.5]
